In [2]:
import re
import numpy as np
import gensim.downloader as api
from datasets import load_dataset
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
dataset = load_dataset("banking77")

In [4]:
X_train_text = dataset["train"]["text"]
y_train_raw = dataset["train"]["label"]

X_test_text = dataset["test"]["text"]
y_test_raw = dataset["test"]["label"]

label_names = dataset["train"].features["label"].names
num_classes = len(label_names)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))
print("Number of intents:", num_classes)

Train size: 10003
Test size: 3080
Number of intents: 77


In [5]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()

train_tokens = [preprocess(t) for t in X_train_text]
test_tokens = [preprocess(t) for t in X_test_text]

In [6]:
w2v = api.load("word2vec-google-news-300")
embedding_dim = 300

In [7]:
vocab = {"<PAD>": 0, "<OOV>": 1}

for sentence in train_tokens:
    for word in sentence:
        if word not in vocab and word in w2v:
            vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 2253


In [8]:
MAX_LEN = 25

def to_seq(tokens):
    return [vocab.get(w, 1) for w in tokens]

X_train = pad_sequences([to_seq(s) for s in train_tokens],
                        maxlen=MAX_LEN, padding="post")

X_test = pad_sequences([to_seq(s) for s in test_tokens],
                       maxlen=MAX_LEN, padding="post")

y_train = to_categorical(y_train_raw, num_classes)
y_test = to_categorical(y_test_raw, num_classes)

In [9]:
embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, idx in vocab.items():
    if word in w2v:
        embedding_matrix[idx] = w2v[word]

In [10]:
model = Sequential([
    Embedding(len(vocab),
              embedding_dim,
              weights=[embedding_matrix],
              input_length=MAX_LEN,
              trainable=False),
    LSTM(128),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/DL/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       675,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 675,900 (2.58 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 675,900 (2.58 MB)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/3


In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=label_names
))

macro_f1 = f1_score(y_true, y_pred, average="macro")
print("Macro F1:", macro_f1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.show()